Data Analyst: Andrei Berner Arroyo Tanta

**Project with Numpy and Pandas 4: Multi-Asset Market Correlation & Beta Analyzer

**Business Application:

This project solves the problem of summarizing millions of price points into a single decision metric. It enables Portfolio Managers to identify "Quiet" vs. "Rebel" assets, facilitating better-balanced portfolios and risk mitigation.

**Executive Summary:

-Key Findings:
Most Volatile ("Rebel"): TSLA showed the highest volatility at 65.29% and a high Beta of 1.78.

Least Volatile ("Quiet"): SPY served as the benchmark with a low volatility of 19.29%.

Relevant Metrics: Analyzed 1,571 rows of multi-asset data (AAPL, MSFT, TSLA, AMZN, GOOGL) to provide a comprehensive view of market sensitivity.

In [11]:
#Setting Enviroment and Data Acquisition & Optimization
import numpy as np
import pandas as pd
import yfinance as yf

In [3]:
#Downloading Multi-Asset Time Series
tickers = "AAPL MSFT TSLA AMZN GOOGL SPY"
data = yf.download(tickers, start="2020-01-01")

#Selection & Memory Engineering
idx = pd.IndexSlice
# We focus on 'Close' prices for return calculation
df_prices = data.loc[:, idx["Close", :]].astype("float32")
# Remove multi-index levels for cleaner manipulation
df_prices.columns = df_prices.columns.get_level_values(1)

print(f"Dataset ingested: {df_prices.shape[0]} rows across {len(df_prices.columns)} assets.")

[*********************100%***********************]  6 of 6 completed

Dataset ingested: 1571 rows across 6 assets.


In [4]:
#Feature Engineering: Daily Returns

#Calculating Daily Returns
#We specify fill_method=None to avoid the FutureWarning and handle NaNs manually
daily_returns = df_prices.pct_change(fill_method=None).fillna(0)

#Descriptive Statistics
print("Summary Statistics of Daily Returns:")
display(daily_returns.describe())

Summary Statistics of Daily Returns:


Ticker,AAPL,AMZN,GOOGL,MSFT,SPY,TSLA
count,1571.000000,1571.000000,1571.000000,1571.000000,1571.000000,1571.000000
mean,0.001001,0.000755,0.001143,0.000747,0.000588,0.002464
std,0.019870,0.022381,0.020310,0.018705,0.012925,0.041358
min,-0.128647,-0.140494,-0.116342,-0.147390,-0.109424,-0.210628
25%,-0.008210,-0.011237,-0.009474,-0.008028,-0.004850,-0.020167
50%,0.001069,0.000711,0.001585,0.000904,0.000900,0.001535
75%,0.010976,0.012954,0.011586,0.010378,0.006711,0.022886
max,0.153288,0.135359,0.102244,0.142169,0.105019,0.226900


In [5]:
#Volatility Analysis (Pandas Layer)

#Annualizing Volatility
#Formula: Daily Std Dev * Square Root of Trading Days (252)
volatility_by_year = daily_returns.groupby(daily_returns.index.year).std() * np.sqrt(252)

#Filtering & Benchmarking
#We exclude the current year to avoid incomplete data bias
volatility_final = volatility_by_year.drop(index=2026).mean()

#Identifying the "Rebel" (Most Volatile) and "Quiet" (Least Volatile) assets
rebel_asset = volatility_final.idxmax()
quiet_asset = volatility_final.idxmin()

print(f"--- Report ---")
print(f"Most Volatile Asset: {rebel_asset} ({volatility_final.max():.2%})")
print(f"Least Volatile Asset: {quiet_asset} ({volatility_final.min():.2%})")

--- Inventory Report ---
Most Volatile Asset: TSLA (65.29%)
Least Volatile Asset: SPY (19.29%)


In [9]:
#The Beta Engine (NumPy Layer)

#Creating the Benchmark (Internal Market)
daily_returns['PORTFOLIO_AVG'] = daily_returns.mean(axis=1)

#Extracting NumPy Arrays for Computation
asset_vals = daily_returns[rebel_asset].values
market_vals = daily_returns['PORTFOLIO_AVG'].values

#Raw Covariance Matrix Calculation
#np.cov returns a matrix: [[Var_A, Cov_AM], [Cov_MA, Var_M]]
cov_matrix = np.cov(asset_vals, market_vals)

raw_covariance = cov_matrix[0, 1]
market_variance = cov_matrix[1, 1]

#Beta Calculation
beta = raw_covariance / market_variance

print(f"--- Report ---")
print(f"--- NumPy Statistical Output ---")
print(f"Asset Analyzed: {rebel_asset}")
print(f"Raw Covariance with Portfolio: {raw_covariance:.6f}")
print(f"Beta Coefficient: {np.round(beta,2)}")

--- Report ---
--- NumPy Statistical Output ---
Asset Analyzed: TSLA
Raw Covariance with Portfolio: 0.000584
Beta Coefficient: 1.78


In [10]:
#Portfolio Correlation Insights
#Generating the Correlation Matrix
correlation_matrix = daily_returns.corr()

print(f"--- Report ---")
#Reporting Internal Market Alignment
print("Asset Alignment with Internal Market (Sorted):")
print(correlation_matrix['PORTFOLIO_AVG'].sort_values(ascending=False))

#Distribution of Annualized Volatility
print("\nDescriptive Summary of Volatility (2020-2025):")
display(volatility_by_year.describe().T)

--- Report ---
Asset Alignment with Internal Market (Sorted):
Ticker
PORTFOLIO_AVG    1.000000
SPY              0.869887
MSFT             0.822063
AAPL             0.812539
GOOGL            0.791323
AMZN             0.789545
TSLA             0.779141
Name: PORTFOLIO_AVG, dtype: float64

Descriptive Summary of Volatility (2020-2025):


,count,mean,std,min,25%,50%,75%,max
Ticker,,,,,,,,
AAPL,7.0,0.296470,0.092571,0.203002,0.236558,0.251019,0.341038,0.466074
AMZN,7.0,0.342294,0.083505,0.240929,0.297812,0.330164,0.364284,0.500773
GOOGL,7.0,0.312301,0.056117,0.243196,0.272691,0.303480,0.353770,0.386506
MSFT,7.0,0.288732,0.087189,0.199615,0.226554,0.250717,0.339581,0.438523
SPY,7.0,0.185594,0.078457,0.125690,0.130321,0.141654,0.218649,0.333873
TSLA,7.0,0.613432,0.156890,0.376432,0.544048,0.634891,0.650857,0.892890
